# Optimizer Trajectory Tree Viewer

Standalone notebook to visualize optimizer trajectories as a tree with:
- trial score per node
- evolved operator (`operator_to_evolve`) shown near each node
- operator-based node colors and legend

Works with trajectory JSON files in `.sempipes_trajectories/` and absolute paths.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
from collections import deque

import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "optimizer_scripts" else Path.cwd()
TRAJ_DIR = REPO_ROOT / ".sempipes_trajectories"

In [ ]:
def list_available_trajectories(pattern: str = "*.json") -> list[Path]:
    if not TRAJ_DIR.exists():
        return []
    return sorted(
        [p for p in TRAJ_DIR.glob(pattern) if "generated_code" not in p.name],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )


def load_trajectory(path: str | Path) -> dict:
    p = Path(path)
    if not p.is_absolute():
        p = TRAJ_DIR / p
    raw = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(raw, str):
        raw = json.loads(raw)
    if not isinstance(raw, dict):
        raise ValueError(f"Unexpected trajectory payload type: {type(raw)}")
    return raw


def _safe_score(value) -> float | None:
    if value is None:
        return None
    try:
        val = float(value)
    except Exception:
        return None
    return val if math.isfinite(val) else None


def parse_outcomes(traj: dict) -> list[dict]:
    parsed = []
    for out in traj.get("outcomes", []):
        sn = out.get("search_node", {})
        trial = sn.get("trial")
        parent = sn.get("parent_trial")
        if trial is None:
            continue
        parsed.append(
            {
                "trial": int(trial),
                "parent_trial": None if parent is None else int(parent),
                "score": _safe_score(out.get("score")),
                "operator_to_evolve": sn.get("operator_to_evolve") or "root",
            }
        )
    parsed.sort(key=lambda x: x["trial"])
    return parsed

In [ ]:
def build_tree(parsed: list[dict]) -> nx.DiGraph:
    G = nx.DiGraph()
    for node in parsed:
        G.add_node(
            node["trial"],
            score=node["score"],
            operator=node["operator_to_evolve"],
        )
        if node["parent_trial"] is not None:
            G.add_edge(node["parent_trial"], node["trial"])
    return G


def _subtree_size(G: nx.DiGraph, node: int) -> int:
    children = sorted(G.successors(node))
    if not children:
        return 1
    return sum(_subtree_size(G, c) for c in children)


def tidy_tree_positions(
    G: nx.DiGraph,
    x_gap: float = 1.3,
    y_gap: float = 1.5,
) -> dict[int, tuple[float, float]]:
    """Tree layout with children centered under parents using subtree widths."""
    roots = sorted([n for n in G.nodes if G.in_degree(n) == 0])
    if not roots:
        return {}

    pos: dict[int, tuple[float, float]] = {}

    def place(node: int, depth: int, x_left: float) -> float:
        children = sorted(G.successors(node))
        if not children:
            x = x_left
            pos[node] = (x, -depth * y_gap)
            return x + x_gap

        cursor = x_left
        child_xs = []
        for child in children:
            cursor_after = place(child, depth + 1, cursor)
            child_xs.append(pos[child][0])
            cursor = cursor_after

        x = sum(child_xs) / len(child_xs)
        pos[node] = (x, -depth * y_gap)
        return cursor

    cursor = 0.0
    for r in roots:
        cursor = place(r, 0, cursor)
        cursor += x_gap

    xs = [p[0] for p in pos.values()]
    center = (min(xs) + max(xs)) / 2 if xs else 0.0
    for n, (x, y) in list(pos.items()):
        pos[n] = (x - center, y)

    return pos

In [ ]:
def best_path_trials(G: nx.DiGraph) -> set[int]:
    best_node = None
    best_score = float("-inf")
    for n, data in G.nodes(data=True):
        score = data.get("score")
        if score is not None and score > best_score:
            best_score = score
            best_node = n

    if best_node is None:
        return set()

    roots = [n for n in G.nodes if G.in_degree(n) == 0]
    if not roots:
        return {best_node}

    best_path = []
    for root in roots:
        try:
            path = nx.shortest_path(G, source=root, target=best_node)
            if not best_path or len(path) < len(best_path):
                best_path = path
        except nx.NetworkXNoPath:
            continue
    return set(best_path)


def draw_trajectory_tree(
    traj: dict,
    fig_size=(24, 14),
    annotate_operator=False,
    highlight_best_path=True,
):
    parsed = parse_outcomes(traj)
    if not parsed:
        raise ValueError("No parseable outcomes found.")

    G = build_tree(parsed)
    pos = tidy_tree_positions(G, x_gap=3.0, y_gap=2.6)

    operators = sorted({data["operator"] for _, data in G.nodes(data=True)})

    # Light pastel palette — text is dark so lighter fills read well.
    palette = [
        "#bfdbfe",  # light blue
        "#fed7aa",  # light orange
        "#ddd6fe",  # light violet
        "#a7f3d0",  # light emerald
        "#fecaca",  # light red
        "#bae6fd",  # light sky
        "#fef08a",  # light yellow
        "#fbcfe8",  # light pink
    ]
    color_map = {op: palette[i % len(palette)] for i, op in enumerate(operators)}

    fig, ax = plt.subplots(figsize=fig_size)
    fig.patch.set_facecolor("#fafafa")
    ax.set_facecolor("#fafafa")

    nx.draw_networkx_edges(
        G,
        pos,
        edge_color="#94a3b8",
        width=1.8,
        arrows=True,
        arrowsize=12,
        ax=ax,
    )

    best_nodes = best_path_trials(G) if highlight_best_path else set()
    if best_nodes:
        best_edges = [(u, v) for (u, v) in G.edges if u in best_nodes and v in best_nodes]
        nx.draw_networkx_edges(
            G,
            pos,
            edgelist=best_edges,
            edge_color="#22c55e",
            width=3.2,
            arrows=True,
            arrowsize=13,
            ax=ax,
        )

    # Color-code each node by evolved operator.
    default_nodes = [n for n in G.nodes if n not in best_nodes]
    if default_nodes:
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=default_nodes,
            node_color=[color_map[G.nodes[n]["operator"]] for n in default_nodes],
            node_size=4200,
            edgecolors="#475569",
            linewidths=2.0,
            node_shape="o",
            ax=ax,
        )
    if best_nodes:
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=list(best_nodes),
            node_color=[color_map[G.nodes[n]["operator"]] for n in best_nodes],
            node_size=4200,
            edgecolors="#22c55e",
            linewidths=3.6,
            node_shape="o",
            ax=ax,
        )

    labels = {}
    for n, data in G.nodes(data=True):
        score = data.get("score")
        labels[n] = "N/A" if score is None else f"{score:.3f}"
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=20, font_weight="bold", font_color="#1e293b", ax=ax)

    legend_items = [
        Patch(facecolor=color_map[op], edgecolor="#334155", label=op)
        for op in operators
    ]
    if best_nodes:
        legend_items.append(Line2D([0], [0], color="#22c55e", lw=3.2, label="Best score path"))

    ax.legend(
        handles=legend_items,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.1),
        ncol=max(2, min(5, len(legend_items))),
        frameon=True,
        facecolor="white",
        edgecolor="#cbd5e1",
        framealpha=0.97,
        fontsize=18,
        title="Operator (node fill color)",
        title_fontsize=18,
    )

    ax.set_title("Optimizer Trajectory Tree", fontsize=24, fontweight="bold", color="#334155", pad=44)
    ax.axis("off")
    fig.tight_layout()
    return G, pos

In [ ]:
# Quick picker
files = list_available_trajectories("fraudbaskets_*.json")
for i, f in enumerate(files[:10]):
    print(f"[{i}] {f.name}")

selected = files[0] if files else None
print("\nSelected:", selected.name if selected else "None")

In [ ]:
if selected is None:
    raise FileNotFoundError("No trajectory files found. Check TRAJ_DIR.")

traj = load_trajectory(selected)
G, pos = draw_trajectory_tree(
    traj,
    fig_size=(24, 14),
    annotate_operator=True,
    highlight_best_path=True,
)
print(f"nodes={G.number_of_nodes()} edges={G.number_of_edges()}")
print("Legend is on top. Node fill color = operator_to_evolve. Best path is green (edges + node borders).")

In [ ]:
# Export last rendered figure
out_dir = REPO_ROOT / "logs"
out_dir.mkdir(parents=True, exist_ok=True)
base = selected.stem if selected is not None else "trajectory"
png_path = out_dir / f"{base}_tree.png"
svg_path = out_dir / f"{base}_tree.svg"

plt.savefig(png_path, dpi=170, bbox_inches="tight")
plt.savefig(svg_path, bbox_inches="tight")
print("Saved:", png_path)
print("Saved:", svg_path)